In [10]:
import pyspark
from pyspark.sql import SparkSession
import sys
import pandas as pd
from pyspark.sql.functions import spark_partition_id

In [2]:
pyspark.__file__

'/Users/osiosman/DEZoomcamp2026/.venv/lib/python3.13/site-packages/pyspark/__init__.py'

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 15:11:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-03-09 15:11:45--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:27cb:ee00:b:20a5:b140:21, 2600:9000:27cb:6000:b:20a5:b140:21, 2600:9000:27cb:aa00:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:27cb:ee00:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M  56.6MB/s    in 1.2s    

2026-03-09 15:11:46 (56.6 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [7]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet') \
    .repartition(4)

In [11]:
df.groupBy(spark_partition_id()).count().show()

+--------------------+-------+
|SPARK_PARTITION_ID()|  count|
+--------------------+-------+
|                   0|1045361|
|                   1|1045361|
|                   2|1045361|
|                   3|1045361|
+--------------------+-------+



In [8]:
df.write.parquet('hw6_output/')

In [12]:
import os
for f in os.listdir('hw6_output/'):
    if f.endswith('.parquet'):
        size_mb = os.path.getsize(f'output/{f}') / (1024 * 1024)
        print(f"{f}: {size_mb:.2f} MB")

part-00002-7b17b67a-eaba-4042-a804-690effdbfeec-c000.snappy.parquet: 24.42 MB
part-00003-7b17b67a-eaba-4042-a804-690effdbfeec-c000.snappy.parquet: 24.41 MB
part-00001-7b17b67a-eaba-4042-a804-690effdbfeec-c000.snappy.parquet: 24.42 MB
part-00000-7b17b67a-eaba-4042-a804-690effdbfeec-c000.snappy.parquet: 24.39 MB


In [13]:
df.show()

[Stage 10:==============================================>         (10 + 2) / 12]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [15]:
from pyspark.sql.functions import col

df.filter(
    (col("tpep_pickup_datetime") >= "2025-11-15 00:00:00") & 
    (col("tpep_pickup_datetime") < "2025-11-16 00:00:00")
).count()


162604

In [19]:
from pyspark.sql.functions import max, unix_timestamp

df.withColumn(
    "trip_hours",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 3600
).orderBy(col("trip_hours").desc()).select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_hours").show(1)


[Stage 34:======================================>                  (8 + 4) / 12]

+--------------------+---------------------+-----------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|       trip_hours|
+--------------------+---------------------+-----------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|90.64666666666666|
+--------------------+---------------------+-----------------+
only showing top 1 row


In [20]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-09 15:45:40--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:27cb:1800:b:20a5:b140:21, 2600:9000:27cb:aa00:b:20a5:b140:21, 2600:9000:27cb:3000:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:27cb:1800:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 15:45:40 (1.15 GB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [21]:
df_zones = spark.read \
    .option("header", "true") \
    .csv("taxi_zone_lookup.csv")

df_zones.createOrReplaceTempView("zones")
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [22]:
df.createOrReplaceTempView("yellow")

In [26]:
spark.sql("""
    SELECT z.Zone, COUNT(*) as cnt
    FROM yellow y
    JOIN zones z ON y.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY cnt ASC
    LIMIT 10
""").show(truncate=False)

[Stage 47:======================================>                  (8 + 4) / 12]

+---------------------------------------------+---+
|Zone                                         |cnt|
+---------------------------------------------+---+
|Eltingville/Annadale/Prince's Bay            |1  |
|Governor's Island/Ellis Island/Liberty Island|1  |
|Arden Heights                                |1  |
|Port Richmond                                |3  |
|Rikers Island                                |4  |
|Rossville/Woodrow                            |4  |
|Great Kills                                  |4  |
|Green-Wood Cemetery                          |4  |
|Jamaica Bay                                  |5  |
|Westerleigh                                  |12 |
+---------------------------------------------+---+

